In [ ]:
import pandas as pd
import numpy as np
import os
import joblib

# ========================================================
# Scenariusze anomalne
# ========================================================

scenarios = {
    "guloader": "../data/zeek_logs/guloader/conn.log",
    "scanning": "../data/zeek_logs/scanning/conn.log",
    "njrat": "../data/zeek_logs/njrat/conn.log",
    "remcos": "../data/zeek_logs/remcos/conn.log",
    "xloader": "../data/zeek_logs/xloader/conn.log",
    "xworm": "../data/zeek_logs/xworm/conn.log",
    "kongtuke1": "../data/zeek_logs/kongtuke1/conn.log",
    "kongtuke2": "../data/zeek_logs/kongtuke2/conn.log",
    "phantomstealer": "../data/zeek_logs/phantomstealer/conn.log",
}

cols = [
    "duration",
    "orig_bytes",
    "resp_bytes",
    "proto",
    "conn_state",
    "orig_pkts",
    "resp_pkts",
]

numeric_cols = ["duration", "orig_bytes", "resp_bytes", "orig_pkts", "resp_pkts"]

# ========================================================
# Wczytanie schematu i scalera z nowego dużego normalnego zbioru
# ========================================================

feature_columns = pd.read_csv("../data/processed/feature_schema.csv")["feature"].tolist()
scaler = joblib.load("../data/processed/standard_scaler.pkl")

print("Liczba cech ze schematu:", len(feature_columns))
print("Pierwsze cechy:", feature_columns[:10])

# ========================================================
# Funkcja czytająca conn.log Zeek
# ========================================================

def read_zeek_conn(path):
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    header = None
    data_lines = []

    for line in lines:
        line = line.strip()

        if not line:
            continue

        if line.startswith("#fields"):
            header = line.split("\t")[1:]
            continue

        if line.startswith("#"):
            continue

        data_lines.append(line.split("\t"))

    if header is None:
        raise ValueError(f"Nie znaleziono #fields w pliku: {path}")

    return pd.DataFrame(data_lines, columns=header)

# ========================================================
# Funkcja preprocessingu zgodna z normalnym zbiorem
# ========================================================

def preprocess_conn(df):
    df2 = df[cols].copy()

    # Konwersja numeryczna
    for col in numeric_cols:
        df2[col] = pd.to_numeric(df2[col], errors="coerce").fillna(0)

    # Transformacja logarytmiczna
    for col in ["duration", "orig_bytes", "resp_bytes"]:
        df2[col] = np.log1p(df2[col])

    # One-hot encoding
    df2 = pd.get_dummies(df2, columns=["proto", "conn_state"], drop_first=True)

    # Dopasowanie do schematu z normalnego zbioru
    df2 = df2.reindex(columns=feature_columns, fill_value=0)

    # Skalowanie scalerem z normalnego zbioru
    X_scaled = scaler.transform(df2)

    return pd.DataFrame(X_scaled, columns=feature_columns)

# ========================================================
# Budowa cech dla wszystkich anomalii
# ========================================================

os.makedirs("../data/processed", exist_ok=True)

summary = []

for name, path in scenarios.items():
    print("\n========================================")
    print("Scenariusz:", name)
    print("Plik:", path)

    df = read_zeek_conn(path)
    print("Flowy surowe:", len(df))

    processed = preprocess_conn(df)

    out_path = f"../data/processed/{name}_features.csv"
    processed.to_csv(out_path, index=False)

    print("Zapisano:", out_path)
    print("Shape:", processed.shape)

    summary.append({
        "scenario": name,
        "raw_flows": len(df),
        "features_shape_rows": processed.shape[0],
        "features_shape_cols": processed.shape[1],
        "output_file": out_path
    })

summary_df = pd.DataFrame(summary)
summary_df.to_csv("../data/processed/anomaly_features_summary.csv", index=False)

print("\n=== PODSUMOWANIE ===")
print(summary_df)
print("\nZapisano summary do: ../data/processed/anomaly_features_summary.csv")